### Credits:

<img align="left" src="https://ithaka-labs.s3.amazonaws.com/static-files/images/tdm/tdmdocs/CC_BY.png"><br />

This notebook is created by [Hannah Jacobs](http://hannahlangstonjacobs.com/) for the [2021 Text Analysis Pedagogy Institute](https://nkelber.github.io/tapi2021/book/intro.html) and then adapted by [Nathan Kelber](http://nkelber.com) under [Creative Commons CC BY License](https://creativecommons.org/licenses/by/4.0/)<br /> 
For questions/comments/improvements, email nathan.kelber@ithaka.org.<br />

Reused and modified for internal use at Università Cattolica del Sacro Cuore di Milano, by Deborah Grbac, email deborah.grbac@unicatt.it and Valentina Schiariti, email valentina.schiariti-collaboratore@unicatt.it, released under CC BY License.

This repository is founded on **Constellate notebooks**. The original Jupyter notebooks repository was designed by the educators at **ITHAKA's Constellate project**. The project was sunset on July 1, 2025. This current repository uses and resuses Constellate notebooks as Open Educational Resources (OER), free for re-use under a Creative Commons CC BY License.
___


# Creating an OCR Workflow (Post-Processing)

These notebooks describe how to turn images and/or pdf documents into plain text using Tesseract optical character recognition. The goal of this notebook is to help users design a workflow for a research project.

**Data Format:** 
* image files (.jpg, .png)
* document files (.pdf)
* plain text (.txt)

**Libraries Used:**
* [Tesseract](https://tesseract-ocr.github.io/) for performing optical character recognition.

**Learning Objectives:**

1. Run OCR on a large batch of prepared images
2. Assess the degree of accuracy achieved in performing OCR
3. Identify post-processing strategies for improving OCR accuracy

**Research Pipeline:**

1. Digitize documents
2. **Optical Character Recognition**
3. Tokenize your texts
4. Perform analysis
___

## "Cleaning" OCR (Post-Processing) Overview

**This part of the process is often best performed with a combination of manual (human) and automated (computer) steps.** This is where you may be addressing not only errors in the OCR itself but also issues with the original printing, as we describe below with regard to hyphenated words at the end of lines. As with pre-processing, how complex you make iterations in this phase depends on your corpus and your resources:

1. **Review the OCR output.** Take an initial look at the OCR text files. Sometimes even just a glance will give you a sense of how well the process has gone. If you see a lot of errors, return to the pre-processing questions and consider which steps you might take to improve the OCR output.


2. **Run a spellchecker & calculate the quality of the OCR output.** Use a spellchecker to get a sense of just how accurate the OCR process may have been. Note that spellchecking here, as with spellchecking in software such as Word, is really looking for known and unknown words.


3. **Use Python to check for and correct possible recurring & unique spelling errors.** These are errors that appear frequently and may be caused by the typescript, hyphenation at the end of lines, or other patterns that Tesseract repeatedly misinterprets. This step should focus on common words and avoid proper nouns (unless you have a full list of proper nouns to draw from). As with any automated step, it's possible that new errors will be introduced here. If there is a known and small quantity of proper nouns used in individual texts or across the corpus, and these are consistently "read" incorrectly by Tesseract, it may be possible to use Python to correct these.


4. If your corpus is small enough and/or you have a team that can help you, **read through the corpus to manually check for and correct unique errors**. This may be a moment to correct proper nouns. If you have a team, it may be advisable to have texts read and corrected by multiple team members. It will be important that these team members have access to both inputs and outputs, and perhaps even lists of proper nouns, to be able to compare the original scans with the computer-readable versions. You may even want to set up a process whereby reviewers can flag words they are not sure about so that another reviewer can provide their opinion so that you and/or another project manager making a final decision on uncertain words.


The above process could be broken down further to address smaller issues incrementally and iteratively. It may also be useful to break your corpus into units of analysis before or during this process to assist with cleaning.

Let's break a PDF down into individual images using the same method from our last lesson.

In [1]:
### Convert a single PDF into a series of image files ###

# Import pdf2image's convert_from_path module.
from pdf2image import convert_from_path
# Import pathlib's Path module.
from pathlib import Path

# Define where the images will be saved
# Check if a folder exists to hold pdfs. If not, create it.
input_folder = Path('./data/sample_pdfs')
input_folder.mkdir(exist_ok=True)

# Get the PDF and convert to a group of PIL (Pillow) objects
# This does NOT save the images as files.
document_path = Path('./data/sample_pdfs/sample_01.pdf')
PIL_objects = convert_from_path(
    document_path,
    poppler_path=r"C:\Users\Utente\tools\poppler\poppler-26.02.0\Library\bin"
)

#define a output folder
output_folder = input_folder / 'pdf_images'
output_folder.mkdir(exist_ok=True)

# For each PIL image object:
for page, image in enumerate(PIL_objects):

    # Create a file name that includes the original file name, and
    # a file number, as well as the file extension.
    fileName = output_folder / f'image_{page}.jpg'

    # Save each PIL image object using the file name created above
    # and declare the image's file format. (Try also PNG or TIFF.)
    image.save(fileName, 'JPEG')

# Success message
print('PDF converted successfully')

PDF converted successfully


And finally, let's batch OCR all the pages, creating a single text file for each image file.

In [2]:
### Convert all the image files into text files ###
import pytesseract

#Import PIL's Image module.
from PIL import Image

input_folder = Path('./data/sample_pdfs/pdf_images')

# For each .jpg file in the input folder, do the following:
for img in input_folder.rglob('*.jpg'):
    # Open the input file and complete OCR
    with open(f'{img}', 'rb') as f_image:
        file = Image.open(f_image)
        ocrText = pytesseract.image_to_string(file)
    
    # Create (or overwrite!) the output file and append the text
    with open(f'{input_folder}/{img.stem}.txt', 'w') as f_text:
        f_text.write(ocrText)

# Post-Processing Step-by-Step

## Review the OCR output.

Open your output text files and begin your review. Make sure to compare them with the original page images. What do you notice?

## Check for misspellings & quality.

Although it appears that this page has been entirely correctly OCR'ed, there are two issues that show up in this text file that we want to address in all of our OCR'ed files:

1. The original printers **broke words at the end of some lines**. For example, `Dis-trict` might be broken up across two lines. How do we deal with this without removing words that *should* be hyphenated?
2. **How would we know how accurate this simple script might be when applied to the entire volume, or to the entire corpus?** 

In addition to being hyphenated, `Dis-trict` may be misspelled as `Dis-triet` or `Dis-trism` in our output—is this just one instance, or does this error recur? If it's recurring, we can use Python to fix it across the corpus. This could be more efficient than having to read the entire OCR'ed corpus. A good starting point is to get a sense of just how accurate the OCR process has been, that is **check its readability**, before we start trying to identify and fix spelling errors.

**In the following scripts, we'll look at how to correct misspelling and check for OCR accuracy by generating a readability score.** During this process, we'll remove the hyphens at the end of lines to help us with spellchecking, but we may find that we introduce new issues for the spellcheck.

To begin, there are a number of modules and libraries we need to import (or reimport) to extend Python's functionality:

In [6]:
### Install PySpellChecker ###
!pip install pyspellchecker


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Import the word_tokenize module from the nltk ("Natural Language Processing Kit") library.
# NLTK is a powerful toolset we can use to manipulate and analyze text data.
import nltk
from nltk import word_tokenize
nltk.download('punkt_tab', download_dir='./data/nltk_data')

[nltk_data] Downloading package punkt_tab to ./data/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
# Import PyTesseract and PIL, an image processing library used by PyTesseract, to complete the OCR.
from PIL import Image
import pytesseract

# Import re, a module that we can use to search text.
import re

# Import glob, a module that helps with file management.
import glob

# Import the SpellChecker module, which we'll use to look for likely misspelled words.
from spellchecker import SpellChecker

# We'll also need the pandas library, which is a powerful toolset for managing data.
# We'll learn more about pandas in the exploratory analysis modules.
import pandas as pd

# This statement confirms that the above code was run without issue.
print("Modules & libraries imported. Ready for the next step.")

Modules & libraries imported. Ready for the next step.


Now we'll set up variables that we'll use to give Python information and structure information that Python returns. These include the location of the original image files and the place we want to store our OCR'ed text, as well as a [spellcheck dictionary](https://pypi.org/project/pyspellchecker/), which we'll extend to include North Carolina placenames, and a dataframe (essentially, an empty table) we'll use to structure readability information along with the OCR'ed text.

*Note: The [spellchecker library](https://pypi.org/project/pyspellchecker/) we are using supports a limited number of Western languages. English is the default.*

In [7]:
# Before we loop through each page, we'll augment our spellchecker 
# dictionary to include place names specific to North Carolina. 
# Our script for gathering these place names is available here: 
# https://github.com/UNC-Libraries-data/OnTheBooks/blob/master/examples/adjustment_recommendation/geonames.py

# Load the spellchecker dictionary.
# Replace the language attribute with another 2 letter code
# to select another language. Options are: English - ‘en’, Spanish - ‘es’,
# French - ‘fr’, Portuguese - ‘pt’, German - ‘de’, Russian - ‘ru’.

spell = SpellChecker(language='en')

# Add the place name words from the "geonames.txt" file to the 
# spellchecker dictionary.
# Sample file to download
geonames_path = Path('./data/geonames.txt')
spell.word_frequency.load_text_file(geonames_path.as_posix())

# This statement confirms that the above code was run without issue.
print("Variables created. Ready for the next step.")

Variables created. Ready for the next step.


Here is what each column will hold:

- **file_name**: The name for the corresponding image file. For now, this is the only information in the table that identifies where the rest of the information in each row comes from (which page).
- **token_count**: The total number of tokens (words) found in each page.
- **unknown_count**: The number of unknown ("misspelled") words found in each page.
- **readability**: Think of this as the percentage of the page that was readable.
- **unknown_words**: A list of tokens (words or in some cases characters) that were not listed in the spellchecker.
- **text**: The OCR'ed text output from each page. The output here includes all <a href="https://en.wikipedia.org/wiki/Escape_character#JavaScript" target="blank">escape characters</a>, so it may look as if a lot of erronenous characters have been added.

Now we'll remove hyphens from the text, run the spellcheck script, and produce a dataframe (table) of information that will give us a sense of the accuracy of our OCR.

In [12]:
### Dictionary Test a Folder of .txt Files ###

# We'll use Pandas to create a dataframe (a table) that can hold 
# information about an OCR'ed page and display it in a tabular format.
# This dataframe will start out empty with only its column headers 
# defined. We'll add information to it one page at a time. So each
# row will represent 1 page.

df = pd.DataFrame(columns=["file_name","token_count","unknown_count","readability","unknown_words","text"])

# Set the folder for the input images
texts_folder = Path('./data/sample_pdfs/pdf_images')

for txt_file in texts_folder.iterdir():
    if txt_file.suffix == '.txt':
    
        # Open each text file and read text into `ocrText`
        with open(txt_file, 'r') as inputFile:
            ocrText = inputFile.read()
            
        # Join hyphenated words that are split between lines by 
        # looking for a hyphen followed by a newline character: "-\n"
        # "\n" is an "escape character" and represents the 
        # "newline," a character that is usually invisible 
        # to human readers but that computers use to mark the 
        # end/beginning of a line. Each time you press the 
        # Enter/Return key on your keyboard, an invisible "\n" 
        # is created to mark the beginning of a new line.
        ocrText = ocrText.replace("-\n","")
        
        # First, we'll use NLTK to "tokenize" text. 
                # "Tokenize" here means to take a page of our OCR'ed text,
                # which Python is currently reading as one big glob of data,
                # and separate each word out so that it can be read as an
                # individual piece of data within a larger data structure 
                # (a list). This process also removes punctuation.
        tokens = word_tokenize(ocrText)
        
        # Lowercase all tokens
        tokens = [token.lower() for token in tokens if token.isalpha()]
        
        # Now we can get all of the words that don't match the 
        # spellchecker dictionary or our list of place names--
        # these are the potential spelling errors.
        unknown = spell.unknown(tokens)
        
        # Let's use a little math to find out how many potential 
        # spelling errors were identified. As part of this process, 
        # we'll create a "readability" score that will give us a 
        # percentage of how readable each file is--how much of the 
        # OCR'ed is "correct."
            
        # If the list of unknown tokens (words) is greater than 0 
        # (i.e. if the list is not empty):
        if len(unknown) != 0:
                
                   # Following order of operations, here's what's happening 
                   # in the readability variable below:
                   # 1. Divide the number of unknown tokens (len(unknown)) 
                        # by the total number of tokens on the page
                        # (len(tokens)). Use "float" to specify that Python
                        # returns a decimal number:
                            # (float(len(unknown))/float(len(tokens))
                   # 2. Multiply the number from step 1 by 100.
                        # (float(len(unknown))/float(len(tokens)) * 100)
                   # 3. Subtract the number from step 2 from 100.
                        # 100 - (float(len(unknown))/float(len(tokens)) * 100)
                   # 4. Round the number from step 3 to 2 decimal places
                        # round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
                
            readability = round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
            
            # If the list of unknown tokens is empty (or equal to 0), then readability is 100!
        else:
            readability = 100
        
        # Let's create a record of the readability information 
        # for this page that we'll add to the dataframe. 
        # The following is a Python dictionary, another way of 
        # storing data. Each word or phrase to the left of the : is a
        # "key" -- think of it as a column header. Each piece of 
        # information to the right is a "value" -- information 
        # written in a single cell below each header. 
    
        df2 = pd.DataFrame({
                "file_name" : txt_file.as_posix(),
                "token_count" : len(tokens),
                "unknown_count" : len(unknown),
                "readability" : readability,
                "unknown_words" : [unknown],
                "text" : ocrText
                })
    
        df = pd.concat([df, df2])
    
        # This statement lets us know if a page has been successfully 
        # checked for readability.
        print(txt_file, "checked for readability.")
    
# This time, instead of creating individual .txt files for each page,
# we're going to save all of the OCR'ed text and readability 
# information to a single .csv ("comma separated value") file. 
# We can view this file format as a table. Having everything stored 
# like this will help us with clean up and future analysis.
df.to_csv(f'{texts_folder}/spellcheck_data.csv', header=True, index=False, sep=',')

# We have the data stored in a file now, but we can also 
# preview it here:
df

# Delete the df variable in case we wish to run this script again
#del df

data\sample_pdfs\pdf_images\image_0-checkpoint.txt checked for readability.
data\sample_pdfs\pdf_images\image_0-checkpoint_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_0.txt checked for readability.
data\sample_pdfs\pdf_images\image_0_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_1.txt checked for readability.
data\sample_pdfs\pdf_images\image_1_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_2.txt checked for readability.
data\sample_pdfs\pdf_images\image_2_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_3.txt checked for readability.
data\sample_pdfs\pdf_images\image_3_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_4.txt checked for readability.
data\sample_pdfs\pdf_images\image_4_corrected.txt checked for readability.
data\sample_pdfs\pdf_images\image_5.txt checked for readability.
data\sample_pdfs\pdf_images\image_5_corrected.txt checked for readability

C:\Users\Utente\AppData\Local\Temp\ipykernel_1528\2812808132.py:94: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df2])


,file_name,token_count,unknown_count,readability,unknown_words,text
0,data/sample_pdfs/pdf_images/image_0-checkpoint...,263,1,99.62,{nontax},SESSION LAWS\n\nOF THE\n\nSTATE OF NORTH CAROL...
0,data/sample_pdfs/pdf_images/image_0-checkpoint...,263,1,99.62,{nontax},SESSION LAWS\n\nOF THE\n\nSTATE OF NORTH CAROL...
0,data/sample_pdfs/pdf_images/image_0.txt,263,1,99.62,{nontax},SESSION LAWS\n\nOF THE\n\nSTATE OF NORTH CAROL...
0,data/sample_pdfs/pdf_images/image_0_corrected.txt,263,1,99.62,{nontax},SESSION LAWS\n\nOF THE\n\nSTATE OF NORTH CAROL...
0,data/sample_pdfs/pdf_images/image_1.txt,389,0,100.00,{},Cu..2-3 1955—-SESSION LAWS\n\nfive thousand do...
0,data/sample_pdfs/pdf_images/image_1_corrected.txt,389,0,100.00,{},Cu..2-3 1955—-SESSION LAWS\n\nfive thousand do...
0,data/sample_pdfs/pdf_images/image_2.txt,420,1,99.76,{subseribe},1955—SESSION LAWS CH. 4\n\nH. B. 34 CHAPTER 4\...
0,data/sample_pdfs/pdf_images/image_2_corrected.txt,420,1,99.76,{subseribe},1955—SESSION LAWS CH. 4\n\nH. B. 34 CHAPTER 4\...
0,data/sample_pdfs/pdf_images/image_3.txt,332,0,100.00,{},CH. 5-6-7 1955—SESSION LAWS\n\nH. B. 2 CHAPTER...
0,data/sample_pdfs/pdf_images/image_3_corrected.txt,332,0,100.00,{},CH. 5-6-7 1955—SESSION LAWS\n\nH. B. 2 CHAPTER...


# Correcting Errors

Broadly speaking, we can break down errors into two categories: **unique** or **recurring**. We can use Python to address both types to an extent, but it's likely that some manual review will still need to be done to ensure the highest quality OCR. Whether and how much manual review can be done will depend on the project's resources.

## Unique Errors

There are at least **two ways to address unique computer-identified errors:**

1. Since we produced a list of unknown words in our readability test, we could simply open each file in a text editor and use find-and-replace functionalities (Command + F or Control + F) to locate and replace instances of unique errors.

2. We could use a little Python to find and replace these errors across the corpus. 

*Caveat: There may be instances where variant spellings are identified as "unknown" (misspelled) but are true representations of the word as it was originally printed. It may be necessary to check these misspellings against the scanned pages and decide whether or not to correct the text in the OCR output.*

The following script runs through the entire sample output (and could be applied to an entire corpus) and checks for and replaces instances of a unique:

In [14]:
# Replacing unknown words with known words
unknown_word = "diseretion"
known_word = "discretion"

# Import glob, a module that helps with file management.
import glob

# Identify the sample_output file path.
# Remember that our readability output is also stored 
# in this file as a .csv. We don't want to change it, 
# so we'll use glob to look for only .txt files.
file_list = glob.glob("./data/sample_pdfs/pdf_images/*.txt")
# Apply the following loop to one file at a time in filePath.

# Read in a file's text
for file in file_list:
    file_path = Path(file)
    with open(file_path) as f:
        text = f.read()
    
    # Correct the unknown word with a known word
    corrected_text = text.replace(unknown_word, known_word)
    with open(file_path, 'w') as f:
        f.write(corrected_text)

print("All instances of " + unknown_word + " replaced with " + known_word + ".")

All instances of diseretion replaced with discretion.


Check the output files for the unknown word to see if the word is still present. 

We've done this for one word at a time, but we could use the list of unknown words generated to create a script that runs through the list and corrects each instance all at once--rather than running the above script for each correction individually.

## Recurring Errors & Changes

There are several kinds of recurring errors:

- Specific Words & Phrases (if a unique mispelling above is present consistently across the corpus, for example).
- Word, Phrase, or Character Patterns (for example, a hyphen used to break up a word at the end of a line).

We looked earlier at how to remove hyphens at the end of lines. To do this we replaced `-\n` with nothing (""). We saved that change to spellcheck csv, but we could have written that to the original text files. We could use the above script to make that change directly in the original output files, though it may be advisable to *keep the original text output files separate from the corrected versions in case you need to refer back.*

We could also use the below script in combination with [regular expressions](https://en.wikipedia.org/wiki/Regular_expression) to correct issues that we know are recurring.

**Be careful when attempting changes with regular expressions**--these always come with the risk of introducing new errors. To avoid as many as possible, make your regular expression as specific as possible.

In [15]:
# Import the regular expressions module (re), 
# which helps us use regex in Python.
import re

# Import glob, a module that helps with file management.
import glob

# Identify the sample_output file path.
# Remember that our readability output is also stored 
# in this file as a .csv. We don't want to change it, 
# so we'll use glob to look for only .txt files.
file_list = glob.glob("./data/sample_pdfs/pdf_images/*.txt")

# Save the pattern for a chapter header (even pages) that we 
# want to search each page for. We've added "^" to our regular 
# expressions to be extra sure that Python searches only at the 
# beginning of each file.
regex_search = re.compile("\n\nThe General Assembly.*?t:\n\n")

# Save the text that we want to use to correct the OCR output.
replacement = "\n\nThe General Assembly of North Carolina do enact:\n\n"

# Apply the following loop to one file at a time in filePath.
for file in file_list:
    
    corrected_file = None # Reset the value of the corrections
    
    file_path = Path(file) 
    with file_path.open() as f:
        text = f.read()
        # Create a corrected version
        corrected_text = re.subn(regex_search, replacement, text)
        print(corrected_text[1], 'match(es) found in ', file_path.name)

    # Write the corrected text to the file
    corrected_file_name = file.replace(".txt", "_corrected.txt")
    with open(corrected_file_name, 'w') as f:
        f.write(corrected_text[0])
    
# The loop will finish when Python has gone through all files in 
# the sample_output folder.

2 match(es) found in  image_0-checkpoint.txt
2 match(es) found in  image_0.txt
1 match(es) found in  image_1.txt
1 match(es) found in  image_2.txt
3 match(es) found in  image_3.txt
2 match(es) found in  image_4.txt
2 match(es) found in  image_5.txt
1 match(es) found in  image_6.txt
1 match(es) found in  image_7.txt
3 match(es) found in  image_8.txt
0 match(es) found in  image_9.txt


# Concatenate all the text files into one
If we are happy with our outputs, then we can stitch all the text files into a single text file.

In [18]:
# Set the folder for the input texts
texts_folder = Path('./data/sample_pdfs/pdf_images')

# Set output filename and create file
full_text = Path('./data/sample_pdfs/full_01.txt')
full_text.touch()

for txt in sorted(texts_folder.rglob('*corrected.txt')):
    with open(txt, 'r') as f_in:
        fileText = f_in.read()
        with open(full_text, 'a') as f_out:
            f_out.write(fileText)

In [23]:

# Read the full text
with open(full_text, 'r') as f:
    text = f.read()

# Print the first 1,000 characters
print(text[:1000])

SESSION LAWS

OF THE

STATE OF NORTH CAROLINA

SESSION 1955

S. B. 4 CHAPTER 1

AN ACT TO AUTHORIZE THE BOARD OF TRUSTEES OF THE
SOUTHERN PINES SCHOOL DISTRICT TO TRANSFER CERTAIN
FUNDS FROM ITS DEBT SERVICE ACCOUNT TO ITS CAPITAL
OUTLAY OR CURRENT EXPENSE ACCOUNTS, OR TO BOTH
SUCH ACCOUNTS.

The General Assembly of North Carolina do enact:

Section 1. The Board of Trustees of the Southern Pines School Dis-
trict is hereby authorized and empowered to transfer all surplus funds held
by it in its debt service account on the date of the ratification of this Act
or on July 1, 1955, to its capital outlay account or current expense account,
or to both such accounts, and to use said funds for capital outlay or current
expense purposes, or both, including the construction of school buildings.

See. 2. All laws and clauses of laws in conflict with this Act are hereby
repealed.

Sec. 3. This Act shall become effective on and after its ratification.

In the General Assembly read three times and r

In [13]:
%pip install pdf2image

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
### Convert a single PDF into a series of image files ###

# Import pdf2image's convert_from_path module.
from pdf2image import convert_from_path
# Import pathlib's Path module.
from pathlib import Path

# # Define where the images will be saved
# # Check if a folder exists to hold pdfs. If not, create it.
data_folder = Path('./data/moby_dick/')
data_folder.mkdir(exist_ok=True, parents=True)

# Get the PDF and convert to a group of PIL (Pillow) objects
# This does NOT save the images as files.
document_path = Path('./data/moby_dick/moby_dick.pdf')
PIL_objects = convert_from_path(
    document_path,
    poppler_path=r"C:\Users\Utente\tools\poppler\poppler-26.02.0\Library\bin"
)

# For each PIL image object:
for page, image in enumerate(PIL_objects):

    # Create a file name that includes the original file name, and
    # a file number, as well as the file extension.
    fileName = f'{data_folder.as_posix()}/image_{str(page)}.jpg'

    # Save each PIL image object using the file name created above
    # and declare the image's file format. (Try also PNG or TIFF.)
    image.save(fileName, 'JPEG')

# Success message
print('PDF converted successfully')

PDF converted successfully


In [15]:
# running OCR 1 try
import pytesseract

#Import PIL's Image module.
from PIL import Image

input_folder = Path('./data/moby_dick')

# For each .jpg file in the input folder, do the following:
for img in input_folder.rglob('*.jpg'):
    # Open the input file and complete OCR
    with open(f'{img}', 'rb') as f_image:
        file = Image.open(f_image)
        ocrText = pytesseract.image_to_string(file)
    
    # Create (or overwrite!) the output file and append the text
    with open(f'{input_folder}/{img.stem}.txt', 'w') as f_text:
        f_text.write(ocrText)

In [16]:
### Install PySpellChecker ###
!pip install pyspellchecker


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# Import re, a module that we can use to search text.
import re
# Import glob, a module that helps with file management.
import glob
# Import the SpellChecker module, which we'll use to look for likely misspelled words.
from spellchecker import SpellChecker
# We'll also need the pandas library, which is a powerful toolset for managing data.
import pandas as pd

# This statement confirms that the above code was run without issue.
print("Modules & libraries imported. Ready for the next step.")

Modules & libraries imported. Ready for the next step.


In [18]:
spell = SpellChecker(language='en')

In [19]:
### Dictionary Test a Folder of .txt Files ###

df = pd.DataFrame(columns=["file_name","token_count","unknown_count","readability","unknown_words","text"])

# Set the folder for the input images
texts_folder = Path('./data/moby_dick')

for txt_file in texts_folder.iterdir():
    if txt_file.suffix == '.txt':
    
        # Open each text file and read text into `ocrText`
        with open(txt_file, 'r') as inputFile:
            ocrText = inputFile.read()
            
        # Join hyphenated words that are split between lines by 
        # looking for a hyphen followed by a newline character: "-\n"
        ocrText = ocrText.replace("-\n","")
        
        # Tokenization with NLTK 
        tokens = word_tokenize(ocrText)
        
        # Lowercase all tokens
        tokens = [token.lower() for token in tokens if token.isalpha()]
        
        # Now we can get all of the words that don't match the 
        # spellchecker dictionary or our list
        unknown = spell.unknown(tokens)
        
        # Let's use a little math to find out how many potential 
        # spelling errors were identified. As part of this process, 
        # we'll create a "readability" score that will give us a 
        # percentage of how readable each file is--how much of the 
        # OCR'ed is "correct."
            
        # If the list of unknown tokens (words) is greater than 0 
        # (i.e. if the list is not empty):
        if len(unknown) != 0:
                
                   # Following order of operations, here's what's happening 
                   # in the readability variable below:
                   # 1. Divide the number of unknown tokens (len(unknown)) 
                        # by the total number of tokens on the page
                        # (len(tokens)). Use "float" to specify that Python
                        # returns a decimal number:
                            # (float(len(unknown))/float(len(tokens))
                   # 2. Multiply the number from step 1 by 100.
                        # (float(len(unknown))/float(len(tokens)) * 100)
                   # 3. Subtract the number from step 2 from 100.
                        # 100 - (float(len(unknown))/float(len(tokens)) * 100)
                   # 4. Round the number from step 3 to 2 decimal places
                        # round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
                
            readability = round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
            
            # If the list of unknown tokens is empty (or equal to 0), then readability is 100!
        else:
            readability = 100
        
        # Let's create a record of the readability information 
        # for this page that we'll add to the dataframe. 
        # The following is a Python dictionary, another way of 
        # storing data. Each word or phrase to the left of the : is a
        # "key" -- think of it as a column header. Each piece of 
        # information to the right is a "value" -- information 
        # written in a single cell below each header. 
    
        df2 = pd.DataFrame({
                "file_name" : txt_file.as_posix(),
                "token_count" : len(tokens),
                "unknown_count" : len(unknown),
                "readability" : readability,
                "unknown_words" : [unknown],
                "text" : ocrText
                })
    
        df = pd.concat([df, df2])
    
        # This statement lets us know if a page has been successfully 
        # checked for readability.
        print(txt_file, "checked for readability.")
    
# This time, instead of creating individual .txt files for each page,
# we're going to save all of the OCR'ed text and readability 
# information to a single .csv ("comma separated value") file. 
# We can view this file format as a table. Having everything stored 
# like this will help us with clean up and future analysis.
df.to_csv(f'{texts_folder}/spellcheck_data.csv', header=True, index=False, sep=',')

# We have the data stored in a file now, but we can also 
# preview it here:
df

C:\Users\Utente\AppData\Local\Temp\ipykernel_1528\985504783.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df2])


data\moby_dick\image_0-checkpoint.txt checked for readability.
data\moby_dick\image_0.txt checked for readability.
data\moby_dick\image_1.txt checked for readability.
data\moby_dick\image_10.txt checked for readability.
data\moby_dick\image_11.txt checked for readability.
data\moby_dick\image_12.txt checked for readability.
data\moby_dick\image_13.txt checked for readability.
data\moby_dick\image_14.txt checked for readability.
data\moby_dick\image_15.txt checked for readability.
data\moby_dick\image_16.txt checked for readability.
data\moby_dick\image_17.txt checked for readability.
data\moby_dick\image_18.txt checked for readability.
data\moby_dick\image_19.txt checked for readability.
data\moby_dick\image_2.txt checked for readability.
data\moby_dick\image_20.txt checked for readability.
data\moby_dick\image_21.txt checked for readability.
data\moby_dick\image_22.txt checked for readability.
data\moby_dick\image_23.txt checked for readability.
data\moby_dick\image_24.txt checked for

,file_name,token_count,unknown_count,readability,unknown_words,text
0,data/moby_dick/image_0-checkpoint.txt,291,7,97.59,"{corlears, moby, loomings, manhattoes, waterwa...",MOBY DICK; OR\nTHE WHITE WHALE\n\nCHAPTER I\n\...
0,data/moby_dick/image_0.txt,291,7,97.59,"{corlears, moby, loomings, manhattoes, waterwa...",MOBY DICK; OR\nTHE WHITE WHALE\n\nCHAPTER I\n\...
0,data/moby_dick/image_1.txt,443,4,99.10,"{moby, saco, inlanders, tranced}",2 MOBY DICK; OR\n\nward. What do you see?—Post...
0,data/moby_dick/image_10.txt,412,14,96.60,"{harpooneer, pha, suphge, thts, goin, ll, il, ...",THE WHITE WHALE il\n\nglasses deceitfully tape...
0,data/moby_dick/image_11.txt,388,10,97.42,"{harpooneer, moby, arrantest, darkcomplexioned...","12 MOBY DICK; OR\n\n“Oh, no,” said he, looking..."
0,data/moby_dick/image_12.txt,419,5,98.81,"{bedwards, alleghanian, harpooneer, bulkington...",THE WHITE WHALE 18\n\nmuch noise as the rest. ...
0,data/moby_dick/image_13.txt,461,9,98.05,"{harpooneer, moby, ll, leaying, draught, could...",14 MOBY DICK; OR\n\nshould tumble in upon me a...
0,data/moby_dick/image_14.txt,359,9,97.49,"{airley, harpooneer, hecla, airth, sartain, tt...","THE WHITE WHALE 15\n\nthink that, after all, I..."
0,data/moby_dick/image_15.txt,460,11,97.61,"{purty, harpooneer, moby, sarmon, goin, tellin...","16 MOBY. DICK; OR\n\none another, and that too..."
0,data/moby_dick/image_16.txt,11,1,90.91,{dodd},"C1\K169281\n\n© Dodd, Mead & Company, Inc.\n\n..."


In [21]:
all_unknown = set()

for words in df["unknown_words"]:
    all_unknown.update(words)

sorted(all_unknown)

['ach',
 'airley',
 'airth',
 'alleghanian',
 'amittai',
 'arrantest',
 'arter',
 'ashbox',
 'balmed',
 'bamboozingly',
 'bedarned',
 'bedwards',
 'behaviour',
 'besmoked',
 'blanco',
 'brighgians',
 'brimmers',
 'bulkington',
 'butcome',
 'cannibalistically',
 'capehorner',
 'catarrhs',
 'catz',
 'centre',
 'civilised',
 'coenties',
 'coftin',
 'colour',
 'comest',
 'corlears',
 'couldn',
 'darkcomplexioned',
 'debel',
 'didn',
 'dishonour',
 'distinetion',
 'dodd',
 'dotings',
 'draught',
 'dreamt',
 'driv',
 'earted',
 'ehrenbreitstein',
 'elephanta',
 'ellery',
 'endeavoured',
 'endeavours',
 'enrertne',
 'erromangoans',
 'ery',
 'eups',
 'farner',
 'favour',
 'favourable',
 'favourite',
 'feejeeans',
 'flourishings',
 'gaspings',
 'gettee',
 'gleig',
 'gogeling',
 'goin',
 'grap',
 'grego',
 'haint',
 'hardicanutes',
 'harpooneer',
 'harpooneers',
 'hatchetfaced',
 'hecla',
 'honour',
 'honourable',
 'hospitalities',
 'ie',
 'ii',
 'iii',
 'il',
 'inlanders',
 'invokingly',
 'iv',

In [31]:
# Add valid Moby-Dick words to the spellchecker's dictionary
spell.word_frequency.load_words([
    "airley",
    "alleghanian",
    "amittai",
    "ashbox",
    "balmed",
    "bamboozingly",
    "bedarned",
    "bedwards",
    "behaviour",
    "besmoked",
    "blanco",
    "brimmers",
    "bulkington",
    "cannibalistically",
    "capehorner",
    "catarrhs",
    "centre",
    "civilised",
    "coenties",
    "colour",
    "comest",
    "corlears",
    "dishonour",
    "dotings",
    "draught",
    "dreamt",
    "ehrenbreitstein",
    "elephanta",
    "ellery",
    "endeavoured",
    "endeavours",
    "erromangoans",
    "favour",
    "favourable",
    "favourite",
    "feejeeans",
    "flourishings",
    "gaspings",
    "hardicanutes",
    "harpooneer",
    "harpooneers",
    "hatchetfaced",
    "hecla",
    "honour",
    "honourable",
    "hospitalities",
    "inlanders",
    "invokingly",
    "kelpy",
    "ledyard",
    "lookest",
    "loomings",
    "maintruck",
    "maketh",
    "manhatto",
    "manhattoes",
    "mapple",
    "marvellous",
    "masoned",
    "mightest",
    "misbehaviour",
    "moby",
    "monkeyjackets",
    "monopolising",
    "nappishness",
    "neighbours",
    "nondescripts",
    "observest",
    "outhanging",
    "pannangians",
    "parlour",
    "placelessly",
    "plaguy",
    "plungings",
    "ponderings",
    "puritanic",
    "purty",
    "queequeg",
    "randolphs",
    "rayther",
    "reasonest",
    "rensselaers",
    "rumour",
    "saco",
    "sailorlike",
    "sartain",
    "sashless",
    "scrutinising",
    "seachest",
    "seafarings",
    "selfcontaining",
    "skrimshander",
    "slantings",
    "squitchy",
    "strainings",
    "supperless",
    "symbolise",
    "tattooings",
    "tongatabooars",
    "tranced",
    "unbecomingness",
    "unbiddenly",
    "uncheered",
    "ungraspable",
    "unhealing",
    "unmethodically",
    "unreluctantly",
    "unstirring",
    "wapping",
    "watchcoats",
    "waterward",
    "whaleman",
    "whalemen",
    "woollen",
    "wrestlings",
    "ii",
    'iii',
    'il',
    'iv',
    'ix',
    'vi',
    'vii',
    'viii',
    "slippering",
    "arrantest",
])
unknown = spell.unknown(tokens)

In [32]:
for word in all_unknown:
    
    for index, row in df.iterrows():
        
        # Convert the text to lowercase so it matches the unknown word
        text = row["text"].lower()
        
        # Find the position of the word
        position = text.find(word)
        
        # If the word is found in this file
        if position != -1:
            
            # Get 100 characters before and after the word
            context = text[position-100:position+len(word)+100]
            
            print("\nFILE:", row["file_name"])
            print("UNKNOWN WORD:", word)
            print("CONTEXT:", context)
            print("-" * 80)


FILE: data/moby_dick/image_14.txt
UNKNOWN WORD: airley
CONTEXT: tily tickled at something beyond my comprehension. ‘no,’ he
answered, “generally he’s an early bird—airley to bed and airley to
rise-—yes, he’s the bird what catches the worm.—but to-night he went
out a-ped
--------------------------------------------------------------------------------

FILE: data/moby_dick/image_24.txt
UNKNOWN WORD: slippering
CONTEXT: her, and suddenly
threw myself at her feet, beseeching her as a particular favour to
give me a good slippering for my misbehaviour; anything indeed but
condemning me to lie abed such an unendurable length of ti
--------------------------------------------------------------------------------

FILE: data/moby_dick/image_10.txt
UNKNOWN WORD: suphge
CONTEXT: , i would put up with
the half of any decent man’s blanket.

“t thought so. all right; take a seat. suphge ¢—you want supper ?
supper’ll be ready directly.” 7

i sat down on an old wooden settle, carved all
--------------

In [ ]:
data/moby_dick/image_10.txt
suphge = supper

In [23]:
del df

#re-run spellcheck 
df = pd.DataFrame(columns=["file_name","token_count","unknown_count","readability","unknown_words","text"])

# Set the folder for the input images
texts_folder = Path('./data/moby_dick')

for txt_file in texts_folder.iterdir():
    if txt_file.suffix == '.txt':
    
        # Open each text file and read text into `ocrText`
        with open(txt_file, 'r') as inputFile:
            ocrText = inputFile.read()
            
        # Join hyphenated words that are split between lines by 
        # looking for a hyphen followed by a newline character: "-\n"
        ocrText = ocrText.replace("-\n","")
        
        # Tokenization with NLTK 
        tokens = word_tokenize(ocrText)
        
        # Lowercase all tokens
        tokens = [token.lower() for token in tokens if token.isalpha()]
        
        # Now we can get all of the words that don't match the 
        # spellchecker dictionary or our list
        unknown = spell.unknown(tokens)
        
        # Let's use a little math to find out how many potential 
        # spelling errors were identified. As part of this process, 
        # we'll create a "readability" score that will give us a 
        # percentage of how readable each file is--how much of the 
        # OCR'ed is "correct."
            
        # If the list of unknown tokens (words) is greater than 0 
        # (i.e. if the list is not empty):
        if len(unknown) != 0:
                
                   # Following order of operations, here's what's happening 
                   # in the readability variable below:
                   # 1. Divide the number of unknown tokens (len(unknown)) 
                        # by the total number of tokens on the page
                        # (len(tokens)). Use "float" to specify that Python
                        # returns a decimal number:
                            # (float(len(unknown))/float(len(tokens))
                   # 2. Multiply the number from step 1 by 100.
                        # (float(len(unknown))/float(len(tokens)) * 100)
                   # 3. Subtract the number from step 2 from 100.
                        # 100 - (float(len(unknown))/float(len(tokens)) * 100)
                   # 4. Round the number from step 3 to 2 decimal places
                        # round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
                
            readability = round(100 - (float(len(unknown))/float(len(tokens)) * 100), 2)
            
            # If the list of unknown tokens is empty (or equal to 0), then readability is 100!
        else:
            readability = 100
        
        # Let's create a record of the readability information 
        # for this page that we'll add to the dataframe. 
        # The following is a Python dictionary, another way of 
        # storing data. Each word or phrase to the left of the : is a
        # "key" -- think of it as a column header. Each piece of 
        # information to the right is a "value" -- information 
        # written in a single cell below each header. 
    
        df2 = pd.DataFrame({
                "file_name" : txt_file.as_posix(),
                "token_count" : len(tokens),
                "unknown_count" : len(unknown),
                "readability" : readability,
                "unknown_words" : [unknown],
                "text" : ocrText
                })
    
        df = pd.concat([df, df2])
    
        # This statement lets us know if a page has been successfully 
        # checked for readability.
        print(txt_file, "checked for readability.")
    
# This time, instead of creating individual .txt files for each page,
# we're going to save all of the OCR'ed text and readability 
# information to a single .csv ("comma separated value") file. 
# We can view this file format as a table. Having everything stored 
# like this will help us with clean up and future analysis.
df.to_csv(f'{texts_folder}/spellcheck_data.csv', header=True, index=False, sep=',')

# We have the data stored in a file now, but we can also 
# preview it here:
df

C:\Users\Utente\AppData\Local\Temp\ipykernel_1528\2503651217.py:77: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df2])


data\moby_dick\image_0-checkpoint.txt checked for readability.
data\moby_dick\image_0.txt checked for readability.
data\moby_dick\image_1.txt checked for readability.
data\moby_dick\image_10.txt checked for readability.
data\moby_dick\image_11.txt checked for readability.
data\moby_dick\image_12.txt checked for readability.
data\moby_dick\image_13.txt checked for readability.
data\moby_dick\image_14.txt checked for readability.
data\moby_dick\image_15.txt checked for readability.
data\moby_dick\image_16.txt checked for readability.
data\moby_dick\image_17.txt checked for readability.
data\moby_dick\image_18.txt checked for readability.
data\moby_dick\image_19.txt checked for readability.
data\moby_dick\image_2.txt checked for readability.
data\moby_dick\image_20.txt checked for readability.
data\moby_dick\image_21.txt checked for readability.
data\moby_dick\image_22.txt checked for readability.
data\moby_dick\image_23.txt checked for readability.
data\moby_dick\image_24.txt checked for

,file_name,token_count,unknown_count,readability,unknown_words,text
0,data/moby_dick/image_0-checkpoint.txt,291,1,99.66,{catz},MOBY DICK; OR\nTHE WHITE WHALE\n\nCHAPTER I\n\...
0,data/moby_dick/image_0.txt,291,1,99.66,{catz},MOBY DICK; OR\nTHE WHITE WHALE\n\nCHAPTER I\n\...
0,data/moby_dick/image_1.txt,443,0,100.00,{},2 MOBY DICK; OR\n\nward. What do you see?—Post...
0,data/moby_dick/image_10.txt,412,11,97.33,"{pha, suphge, thts, goin, ll, haint, didn, pie...",THE WHITE WHALE il\n\nglasses deceitfully tape...
0,data/moby_dick/image_11.txt,388,3,99.23,"{darkcomplexioned, arrantest, ll}","12 MOBY DICK; OR\n\n“Oh, no,” said he, looking..."
0,data/moby_dick/image_12.txt,419,0,100.00,{},THE WHITE WHALE 18\n\nmuch noise as the rest. ...
0,data/moby_dick/image_13.txt,461,4,99.13,"{couldn, leaying, ve, ll}",14 MOBY DICK; OR\n\nshould tumble in upon me a...
0,data/moby_dick/image_14.txt,359,4,98.89,"{airley, tt, couldn, airth}","THE WHITE WHALE 15\n\nthink that, after all, I..."
0,data/moby_dick/image_15.txt,460,7,98.48,"{sarmon, goin, tellin, sellin, airth, ve, butc...","16 MOBY. DICK; OR\n\none another, and that too..."
0,data/moby_dick/image_16.txt,11,1,90.91,{dodd},"C1\K169281\n\n© Dodd, Mead & Company, Inc.\n\n..."
